In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Procore — Prime Contract Line Items
# MAGIC For every project in the company, pulls all Prime Contracts, then pulls
# MAGIC all Line Items (including Cost Code) for each contract and saves raw
# MAGIC JSON to the Bronze lakehouse.

# COMMAND ----------

import requests
import json
import time
import datetime
from pyspark.sql import Row

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Auth — run the existing procore_auth notebook to get a token

# COMMAND ----------

auth_result = mssparkutils.notebook.run("procore_auth", 90)
auth_data = json.loads(auth_result)

ACCESS_TOKEN = auth_data["token"]
COMPANY_ID = auth_data["company_id"]

BASE_URL = "https://api.procore.com"

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Procore-Company-Id": str(COMPANY_ID),
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Look up all projects in the company

# COMMAND ----------

projects_resp = requests.get(
    f"{BASE_URL}/rest/v1.0/projects",
    headers=HEADERS,
    params={"company_id": COMPANY_ID, "serializer_view": "compact", "per_page": 200},
)
projects_resp.raise_for_status()
all_projects = projects_resp.json()
project_ids = [p["id"] for p in all_projects]
print(f"Found {len(project_ids)} project(s)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Helper: GET with pagination and 429 retry/backoff

# COMMAND ----------

def get_all_pages(url, params=None, per_page=100, max_retries=5):
    params = dict(params or {})
    params["per_page"] = per_page

    all_results = []
    page = 1
    while True:
        params["page"] = page

        retries = 0
        while True:
            resp = requests.get(url, headers=HEADERS, params=params)

            if resp.status_code == 429:
                retries += 1
                if retries > max_retries:
                    resp.raise_for_status()
                wait_seconds = int(resp.headers.get("Retry-After", 30))
                print(f"  -> 429 rate limited, waiting {wait_seconds}s (retry {retries}/{max_retries})...")
                time.sleep(wait_seconds)
                continue

            if not resp.ok:
                print(f"  -> {resp.status_code} on {resp.url}")
                print(f"  -> body: {resp.text[:500]}")
            resp.raise_for_status()
            break

        batch = resp.json()

        if not isinstance(batch, list):
            raise ValueError(f"Expected a list response, got: {type(batch)} -> {batch}")

        all_results.extend(batch)

        if len(batch) < per_page:
            break
        page += 1

    return all_results

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Helper: pull prime contracts + line items for one project

# COMMAND ----------

def get_prime_contract_line_items_for_project(project_id):
    contracts_url = f"{BASE_URL}/rest/v1.0/prime_contracts"
    try:
        contracts = get_all_pages(contracts_url, params={"project_id": project_id})
    except requests.HTTPError as e:
        print(f"  [project {project_id}] skipped — {e}")
        return []

    project_line_items = []
    for contract in contracts:
        contract_id = contract["id"]
        line_items_url = f"{BASE_URL}/rest/v1.0/prime_contracts/{contract_id}/line_items"
        try:
            line_items = get_all_pages(line_items_url, params={"project_id": project_id})
        except requests.HTTPError as e:
            print(f"  [project {project_id}, contract {contract_id}] skipped — {e}")
            continue

        for li in line_items:
            li["_project_id"] = project_id
            li["_prime_contract_id"] = contract_id
            li["_prime_contract_title"] = contract.get("title")

        project_line_items.extend(line_items)

    return project_line_items

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Loop over every project

# COMMAND ----------

all_line_items = []

for project_id in project_ids:
    print(f"Project {project_id}...")
    line_items = get_prime_contract_line_items_for_project(project_id)
    print(f"  -> {len(line_items)} line item(s)")
    all_line_items.extend(line_items)

print(f"\nTotal line items pulled: {len(all_line_items)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Write raw results to Bronze lakehouse

# COMMAND ----------

BRONZE_TABLE = "procore_prime_contract_line_items"  # resolves against attached default lakehouse

pull_ts = datetime.datetime.utcnow().isoformat()

bronze_rows = [
    Row(
        project_id=li.get("_project_id"),
        prime_contract_id=li.get("_prime_contract_id"),
        prime_contract_title=li.get("_prime_contract_title"),
        line_item_id=li.get("id"),
        raw_json=json.dumps(li),
        pulled_at=pull_ts,
    )
    for li in all_line_items
]

if bronze_rows:
    bronze_df = spark.createDataFrame(bronze_rows)
    bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"Wrote {bronze_df.count()} row(s) to {BRONZE_TABLE}")
else:
    print("No line items pulled this run — nothing written to bronze.")

StatementMeta(, 15520384-096a-4e65-9e40-cc8c4eba6a02, 3, Finished, Available, Finished, False)

Found 18 project(s)
Project 562949955001573...
  -> 15 line item(s)
Project 562949954833574...
  -> 57 line item(s)
Project 562949955118102...
  -> 11 line item(s)
Project 562949955225798...
  -> 72 line item(s)
Project 562949955257421...
  -> 0 line item(s)
Project 562949955064640...
  -> 31 line item(s)
Project 562949955318524...
  -> 0 line item(s)
Project 562949955286476...
  -> 12 line item(s)
Project 562949955375634...
  -> 26 line item(s)
Project 562949954973730...
  -> 1 line item(s)
Project 562949954973684...
  -> 1 line item(s)
Project 562949954971827...
  -> 7 line item(s)
Project 562949953807489...
  -> 6 line item(s)
Project 562949955365267...
  -> 31 line item(s)
Project 562949953807474...
  -> 0 line item(s)
Project 562949954507004...
  -> 40 line item(s)
Project 562949955173068...
  -> 404 on https://api.procore.com/rest/v1.0/prime_contracts?project_id=562949955173068&per_page=100&page=1
  -> body: {"errors":"Not found"}
  [project 562949955173068] skipped — 404 Client 